# Tracing & Observability for LLM Systems: A Hands-On Lab

**A hands-on lab:** debug a real customer support bot using 5 tracing approaches.

**The story:** you are the AI engineer at **CloudNest**, a cloud-storage startup. Its support bot answers questions from the real product docs. One morning a customer tweets:

> *"@CloudNest your bot told me my free plan gives 5 GB storage. Your pricing page says 15 GB!"*

Your manager asks just one question: **"Why did the bot say 5?"**

Without observability, you'd spend 3 hours guessing. With it, you'll answer in 30 seconds. Let's see how.

| Part | Theory from the notes | Tool | Real use case |
|---|---|---|---|
| 0 | Setup | - | - |
| 1 | Step 0: why LLM testing is harder | Plain Python | 5 answers that all "pass" a test |
| 2 | Step 1: span, trace, tree of spans | Our own tracer, from scratch | Find out why the bot said 5 |
| 3 | Tool 1 | LangSmith | Multi-step LangGraph agent with tools |
| 4 | Tool 2 | Langfuse | Non-LangChain chatbot, tracked per customer and session |
| 5 | Tool 3 | OpenTelemetry + OpenInference | Vendor-neutral spans, the raw format |
| 6 | Tool 4 | Arize Phoenix | RAG debugging at scale, before vs. after the fix |
| 7 | Summary | - | Which tool to use when |

## Part 0: Setup

In [ ]:
# Bring the api_key from secrets into a variable
from google.colab import userdata

my_api_key = userdata.get('agentic_ai')

In [ ]:
pip install langchain-groq langgraph groq sentence-transformers langfuse opentelemetry-sdk openinference-instrumentation-langchain arize-phoenix -q # q for silent installation

In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model = 'openai/gpt-oss-20b',
    temperature = 0, # Creativity level of the model
    api_key = my_api_key,
    max_retries = 6  # retry automatically if Groq's rate limit is reached
)

**Accounts needed for Parts 3 and 4 (free, no cost):**
1. **LangSmith:** sign up at smith.langchain.com -> copy an API key -> save as Colab secret `langsmith_api_key`.
2. **Langfuse:** sign up at cloud.langfuse.com -> copy your public + secret key -> save as Colab secrets `langfuse_public_key` and `langfuse_secret_key`.

Parts 2, 5 and 6 need no account (our own tracer, raw OpenTelemetry, and a locally-run Phoenix).

### CloudNest's real docs (used throughout this notebook)

Four short sections. Notice the first one: a **discontinued** plan that used to be called "Free" before a 2023 rebrand - wording close enough to the *current* Free plan that a retriever can genuinely confuse the two.

In [ ]:
chunks = {
    'legacy_starter': ("CloudNest Starter plan (discontinued in 2023, previously marketed as the "
                        "'Free' tier before the rebrand): 5 GB storage, no file versioning, "
                        "single device sync only."),
    'current_free':   ("CloudNest Free plan (current, active since 2023): 15 GB storage, "
                        "30-day file versioning, sync across up to 3 devices."),
    'pro':            ("CloudNest Pro plan: 2 TB storage, 1-year file versioning, unlimited "
                        "devices. $9/month."),
    'business':       ("CloudNest Business plan: 5 TB storage per user, unlimited versioning, "
                        "admin controls. $15 per user per month."),
}

## Part 1: Step 0 - why LLM testing is harder than normal software

Our notes' point: *"is this a good answer?" is not binary*. To see this for real, we generate 5 genuinely different answers to the same question - one per chunk given as context, plus one with no context at all - and run a **naive pass/fail test** on all of them.

In [ ]:
question = "How much storage does CloudNest's free plan include?"

def answer_with_context(context):
    prompt = f"""Answer the question using the context below.

Context: {context}

Question: {question}"""
    return llm.invoke(prompt).content

answers = {name: answer_with_context(text) for name, text in chunks.items()}
answers['no_context'] = llm.invoke(question).content

for name, answer in answers.items():
    print(f'[{name}]', answer)
    print()

In [ ]:
# A naive test: does the answer mention a number, "GB", and the word "plan"?
import re

def naive_pass(answer):
    return bool(re.search(r'\d+\s*GB', answer)) and 'plan' in answer.lower()

for name, answer in answers.items():
    print(f"{name}: {'PASS' if naive_pass(answer) else 'FAIL'}")

**What to look for:** the answer built from `legacy_starter` (wrong: 5 GB) and the one from `current_free` (correct: 15 GB) both mention a number, "GB" and "plan" - so they both **pass** the naive test. A test that only checks shape, not correctness, cannot catch this. This is exactly our notes' point, made concrete with real model output.

## Part 2: Step 1 - spans, a trace, a tree of spans (built from scratch)

A **span** records one operation: start time, end time, input, output. Spans **nest** into a **trace** - here, a retrieval span inside the overall request, feeding an LLM-call span. We build this ourselves first, with no library, so the idea is fully transparent before we reach for any tool.

In [ ]:
import time

class Span:
    def __init__(self, name, parent=None):
        self.name, self.parent, self.children = name, parent, []
        self.input = self.output = None
        self.start_time = time.time()
        if parent:
            parent.children.append(self)

    def end(self, input=None, output=None):
        self.input, self.output, self.end_time = input, output, time.time()

    def print_tree(self, indent=0):
        duration = (self.end_time - self.start_time) * 1000
        print(' ' * indent + f"- {self.name} ({duration:.0f} ms)")
        if self.input is not None:
            print(' ' * (indent + 2) + f"input:  {str(self.input)[:100]}")
        if self.output is not None:
            print(' ' * (indent + 2) + f"output: {str(self.output)[:100]}")
        for child in self.children:
            child.print_tree(indent + 4)

### A real (dense) retriever over the 4 CloudNest chunks

Real embeddings, real cosine similarity - this is the actual mechanism that can retrieve the wrong chunk.

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

embedder = SentenceTransformer('all-MiniLM-L6-v2')
chunk_names, chunk_texts = list(chunks.keys()), list(chunks.values())
chunk_embeddings = embedder.encode(chunk_texts, normalize_embeddings=True)

def retrieve(query):
    query_embedding = embedder.encode(query, normalize_embeddings=True)
    scores = chunk_embeddings @ query_embedding
    best = int(np.argmax(scores))
    return chunk_names[best], chunk_texts[best], dict(zip(chunk_names, scores.round(3)))

### Run the real customer question through our own tracer

In [ ]:
request_span = Span('customer_request')

retrieval_span = Span('retrieval', parent=request_span)
chosen_name, chosen_text, all_scores = retrieve(question)
retrieval_span.end(input=question, output=f'{chosen_name}: {chosen_text}')

llm_span = Span('llm_call', parent=request_span)
final_answer = answer_with_context(chosen_text)
llm_span.end(input=chosen_text, output=final_answer)

request_span.end(input=question, output=final_answer)

print('Similarity scores for every chunk:', all_scores)
print()
request_span.print_tree()

**This answers the manager's question directly:** the `retrieval` span's `output` line names exactly which chunk was pulled, and the similarity scores show how close the call was. If `legacy_starter` won, you now know *precisely* why the bot said 5 - not by guessing, but by reading the trace. If `current_free` won instead on your run, that's useful too: it means this particular question didn't trigger the bug, and you can try rewording it closer to the real customer's tweet.

## Part 3: LangSmith - a multi-step LangGraph agent with tools

Our notes' description of LangSmith: *"set an environment variable and traces appear."* We build a real two-tool LangGraph agent, then trace it with **zero changes to the agent's own code**.

In [ ]:
def account_lookup(customer_id: str) -> str:
    """Look up a CloudNest customer's current plan."""
    accounts = {'C1001': 'Free plan', 'C1002': 'Pro plan'}
    return accounts.get(customer_id, 'Unknown customer')

def docs_lookup(query: str) -> str:
    """Search CloudNest's product docs for an answer to a plan or storage question."""
    _, text, _ = retrieve(query)
    return text

In [ ]:
from typing import Annotated, Literal
from typing_extensions import TypedDict
from langgraph.graph.message import add_messages
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode
from langchain_core.messages import ToolMessage

class State(TypedDict):
    messages: Annotated[list, add_messages]

agent_llm = llm.bind_tools([account_lookup, docs_lookup])

def assistant_node(state):
    return {'messages': [agent_llm.invoke(state['messages'])]}

def should_continue(state) -> Literal['tools', END]:
    return 'tools' if state['messages'][-1].tool_calls else END

builder = StateGraph(State)
builder.add_node('assistant', assistant_node)
builder.add_node('tools', ToolNode([account_lookup, docs_lookup]))
builder.add_edge(START, 'assistant')
builder.add_conditional_edges('assistant', should_continue)
builder.add_edge('tools', 'assistant')
support_graph = builder.compile()

In [ ]:
import os

# This is the ONLY setup step - the agent above is not touched at all
os.environ['LANGCHAIN_TRACING_V2'] = 'true'
os.environ['LANGCHAIN_API_KEY'] = userdata.get('langsmith_api_key')
os.environ['LANGCHAIN_PROJECT'] = 'cloudnest-support-bot'

In [ ]:
result = support_graph.invoke({'messages': [('user', 'Customer C1002 is asking how much storage their plan includes.')]})
print(result['messages'][-1].content)

**What to check:** open your LangSmith project (smith.langchain.com) - you should see this run as a trace with the `assistant` -> `tools` -> `assistant` steps nested, including both tool calls and the final LLM call.

## Part 4: Langfuse - a non-LangChain chatbot, tracked per customer and session

Our notes: *"framework agnostic."* To prove that for real, this chatbot uses the **raw Groq SDK directly** - no LangChain anywhere - and we attach a customer ID and session ID to every trace, so support tickets can be filtered per customer.

In [ ]:
from groq import Groq

raw_client = Groq(api_key=my_api_key)

def raw_chat(prompt):
    response = raw_client.chat.completions.create(
        model='openai/gpt-oss-20b',
        messages=[{'role': 'user', 'content': prompt}],
    )
    return response.choices[0].message.content

In [ ]:
import os

# v3 SDK reads credentials from environment variables
os.environ['LANGFUSE_PUBLIC_KEY'] = userdata.get('langfuse_public_key')
os.environ['LANGFUSE_SECRET_KEY'] = userdata.get('langfuse_secret_key')
os.environ['LANGFUSE_HOST'] = 'https://cloud.langfuse.com'

In [ ]:
from langfuse import get_client

langfuse = get_client()

In [ ]:
def traced_chat(customer_id, session_id, prompt):
    with langfuse.start_as_current_observation(
        as_type='span',
        name='cloudnest-support-chat',
        input=prompt,
        metadata={'langfuse_user_id': customer_id, 'langfuse_session_id': session_id},
    ) as span:
        with langfuse.start_as_current_observation(
            as_type='generation', name='raw-groq-call', model='openai/gpt-oss-20b', input=prompt
        ) as generation:
            answer = raw_chat(prompt)
            generation.update(output=answer)
        span.update(output=answer)
    return answer

In [ ]:
# Two different CloudNest customers, each in their own session
print(traced_chat('C1001', 'session-A', 'How much storage do I get on the Free plan?'))
print(traced_chat('C1002', 'session-B', 'What does the Pro plan include?'))

langfuse.flush()  # make sure both traces are sent before we check the dashboard

**What to check:** open your Langfuse project (cloud.langfuse.com) and filter traces by `user_id` - you should see `C1001` and `C1002` as separate, filterable customers, with no LangChain involved anywhere in this section.

## Part 5: OpenTelemetry + OpenInference - vendor-neutral spans, the raw format

Our notes: *"instrument once, send to any backend."* We reuse the **same** OpenInference instrumentor Phoenix uses in Part 6 - but this time we point it at a plain console exporter instead of any dashboard, to see the actual raw span format underneath every tool above.

In [ ]:
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import SimpleSpanProcessor, ConsoleSpanExporter
from openinference.instrumentation.langchain import LangChainInstrumentor

otel_provider = TracerProvider()
otel_provider.add_span_processor(SimpleSpanProcessor(ConsoleSpanExporter()))
LangChainInstrumentor().instrument(tracer_provider=otel_provider)

In [ ]:
print(llm.invoke('In one sentence, what is CloudNest?').content)

**What to look for in the raw output above:** a JSON-like span with fields like `name`, `attributes` (model name, token counts), `start_time`, `end_time`. This is the **same shape of data** Phoenix, LangSmith and Langfuse all built their dashboards on top of - here it is with no dashboard at all, just the open standard.

## Part 6: Arize Phoenix - RAG debugging at scale, before vs. after the fix

Part 2 found the bug on **one** question. Real support bots get many questions at once. Here we run a **batch** of 5 real customer questions through Phoenix, see the wrong-chunk pattern at scale, then genuinely **fix** the retriever and compare.

In [ ]:
import phoenix as px

session = px.launch_app()  # a real, local tracing server - no account needed

In [ ]:
from phoenix.otel import register

phoenix_provider = register(endpoint=session.url + 'v1/traces', project_name='cloudnest-rag')
LangChainInstrumentor().instrument(tracer_provider=phoenix_provider)  # re-point the SAME instrumentor at Phoenix

In [ ]:
customer_questions = [
    "How much storage does the free plan include?",
    "What's the storage limit if I'm not paying anything?",
    "I'm on the no-cost tier, how many GB do I get?",
    "Does the Pro plan include file versioning?",
    "How many devices can sync on the Business plan?",
]

def run_rag(question):
    chosen_name, _, _ = retrieve(question)
    context = chunks[chosen_name]
    return chosen_name, answer_with_context(context)

In [ ]:
print('BEFORE the fix:\n')
for question in customer_questions:
    chosen_name, answer = run_rag(question)
    print(f'[{chosen_name}] {question}\n  -> {answer}\n')

### Look at the pattern in Phoenix, then fix it for real

A real fix: tag the discontinued plan clearly as **not current**, right at the start of the chunk text, so a real retriever has a much stronger signal to avoid it for "free plan" questions phrased in the present tense.

In [ ]:
chunks['legacy_starter'] = ("[DISCONTINUED PLAN - do not use for current pricing questions] " + chunks['legacy_starter'])
chunk_texts = list(chunks.values())
chunk_embeddings = embedder.encode(chunk_texts, normalize_embeddings=True)  # re-embed with the fixed text

In [ ]:
print('AFTER the fix:\n')
for question in customer_questions:
    chosen_name, answer = run_rag(question)
    print(f'[{chosen_name}] {question}\n  -> {answer}\n')

In [ ]:
session.view()  # compare both batches as real traces inside Phoenix

**What to look for:** count how many of the 5 questions picked `legacy_starter` before the fix vs. after. A real improvement here is a real drop in that count - not a guaranteed drop to zero, since embedding similarity is continuous, not a hard rule.

## Part 7: Summary - which tool to use when

| Situation | Tool | Why |
|---|---|---|
| You just need to see what happened once, no setup overhead | Our own tracer (Part 2) | Full control, zero dependencies, good for understanding the concept |
| Your agent is already built in LangGraph/LangChain | LangSmith | Native integration - literally just env variables |
| Your app does NOT use LangChain, or you need per-customer/per-session views | Langfuse | Framework agnostic, open source, self-hostable |
| You need a vendor-neutral format you can send to any backend later | OpenTelemetry + OpenInference | The standard underneath Phoenix, and increasingly LangSmith/Langfuse too |
| You're debugging RAG retrieval quality across many queries at once | Arize Phoenix | Native RAGAS-style evals, batch comparison, no account needed |

**The manager's question, answered:** "Why did the bot say 5?" - a wrong chunk, retrieved because its wording was too close to the right one. Every tool above can show you that same fact; which one you reach for depends on what you've already built and who else needs to see it.